In [1]:
import sys
sys.path.append("/Users/avelezxerenity/Documents/GitHub/pysdk")
#sys.path.append("/Users/andre/Documents/xerenity/pysdk")
import numpy as np
import numpy_financial as npf
from scipy.optimize import newton
import os
import json
from src.xerenity.xty import Xerenity
from server.loan_calculator.loan_calculator import LoanCalculatorServer
from utilities.date_functions import datetime_to_ql,ql_to_datetime
import pandas as pd
import QuantLib as ql
from loans_calculator.funciones_analisis_credito import calculate_days_from_value_date,create_cashflows_and_total_value
from  datetime import datetime
from utilities.date_functions import days_30_360_dt,days_30_360_ql,days_act_act_dt,days_act_act_ql,days_act_365_ql,days_act_365_dt
xty = Xerenity(
    username=os.getenv('XTY_USER'),
    password=os.getenv('XTY_PWD'),
)

all_loans_data = xty.get_all_loan_data(
    filter_date="2024-08-23"
)


ibr_cashflow = xty.get_ibr_data(
    loan_id="beb65020-e940-4995-9e1e-2f2208cdc638",
    filter_date="2024-08-23"
)
calc = LoanCalculatorServer(ibr_cashflow, local_dev=True)
loan_payments = calc.cash_flow_ibr()


# Define the value_date

value_date=datetime_to_ql(datetime.strptime(all_loans_data['filter_date'], '%Y-%m-%d'))
db_info=all_loans_data['db_info']

2024-08-30 18:40:01,182:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/auth/v1/token?grant_type=password "HTTP/1.1 200 OK"
2024-08-30 18:40:01,723:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/rpc/ibr_cash_flow_data_all "HTTP/1.1 200 OK"
2024-08-30 18:40:02,065:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/rpc/ibr_cash_flow_data "HTTP/1.1 200 OK"
/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '83333333.33333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments


In [2]:
days_converter_dir={'por_dias_360':'30/360','por_dias_365':'actual/365'}
db_info=all_loans_data['db_info']

In [3]:
# Initialize an empty dictionary to store results
results = {}

# Define a unique identifier for each loan (e.g., a code or index)
for i, loan in enumerate(all_loans_data['loans']):
    try:
        # Create a copy of the current loan dictionary
        loan_temp = loan.copy()
        
        # Add or update 'db_info' in the copied dictionary
        loan_temp['db_info'] = db_info
        
        # Instantiate the LoanCalculatorServer with the updated dictionary
        calc = LoanCalculatorServer(loan_temp, local_dev=True)
        
        # Calculate loan payments
        loan_payments = calc.cash_flow_ibr()
        
        # Create cash flows and total value
        variables = create_cashflows_and_total_value(
            pd.DataFrame(loan_payments),
            value_date,
            days_converter_dir[loan['days_count']]
        )
        
        # Save the result in the dictionary with a unique code (e.g., 'loan_1', 'loan_2', ...)
        results[f'loan_{i}'] = variables
        
    except Exception as e:
        # Handle any unexpected exceptions
        print(f"An unexpected error occurred: {e}")
        print(loan)

# Print results to verify
print(results)



/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '83333333.33333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

An unexpected error occurred: unsupported operand type(s) for *: 'int' and 'NoneType'
{'days_count': 'por_dias_360', 'grace_type': 'capital', 'start_date': '2024-12-28', 'periodicity': 'Mensual', 'grace_period': 36, 'interest_rate': 8.5, 'min_period_rate': None, 'original_balance': 313000000, 'number_of_payments': 84}
An unexpected error occurred: unsupported operand type(s) for *: 'int' and 'NoneType'
{'days_count': 'por_dias_360', 'grace_type': 'capital', 'start_date': '2025-04-30', 'periodicity': 'Mensual', 'grace_period': 24, 'interest_rate': 8.5, 'min_period_rate': None, 'original_balance': 387000000, 'number_of_payments': 72}
{'loan_0': {'cashflows':         date       payment
0 2024-08-23 -5.874901e+08
1 2024-09-06  9.066875e+07
2 2024-10-06  8.960027e+07
3 2024-11-06  8.827145e+07
4 2024-12-06  8.734139e+07
5 2025-01-06  8.609435e+07
6 2025-02-06  8.517401e+07
7 2025-03-06  8.430632e+07, 'irr': 0.15589199839423032, 'duration': 0.24301047290599415, 'previous_payment_date': Times

/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '83333333.33333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

In [4]:
results

{'loan_0': {'cashflows':         date       payment
  0 2024-08-23 -5.874901e+08
  1 2024-09-06  9.066875e+07
  2 2024-10-06  8.960027e+07
  3 2024-11-06  8.827145e+07
  4 2024-12-06  8.734139e+07
  5 2025-01-06  8.609435e+07
  6 2025-02-06  8.517401e+07
  7 2025-03-06  8.430632e+07,
  'irr': 0.15589199839423032,
  'duration': 0.24301047290599415,
  'previous_payment_date': Timestamp('2024-08-06 00:00:00'),
  'days_to_previous': -17,
  'next_payment_date': Timestamp('2024-09-06 00:00:00'),
  'days_to_next': 13,
  'next_interest_payment': 7335416.666666665,
  'accrued_interest': 4156736.1111111105},
 'loan_1': {'cashflows':          date       payment
  0  2024-08-23 -9.479590e+08
  1  2024-09-14  3.988349e+07
  2  2024-10-14  3.975876e+07
  3  2024-11-14  3.853341e+07
  4  2024-12-14  3.850087e+07
  5  2025-01-14  3.751154e+07
  6  2025-02-14  3.627650e+07
  7  2025-03-14  3.714131e+07
  8  2025-04-14  3.555372e+07
  9  2025-05-14  3.566078e+07
  10 2025-06-14  3.497772e+07
  11 2025-0

In [ ]:
#
# for loan in all_loans_data['loans']:

calc = LoanCalculatorServer(loan, local_dev=True)
loan_payments = calc.cash_flow_ibr()
variables=create_cashflows_and_total_value(pd.DataFrame(loan_payments), value_date, days_converter_dir[loan['days_count']])

In [ ]:
\





In [ ]:

variables


In [ ]:
type(variables)

In [ ]:
create_cashflows_and_total_value(loan_payments, value_date, '30/360')

In [ ]:
calculate_debt_duration(loan_payments)

In [ ]:
flows.to_clipboard()

In [ ]:
calculate_irr(flows['date'],flows['payment'],'30/360')*100

In [ ]:
loan_payments.to_clipboard()